# 04 — Resultados económicos

Explora los resultados de F3: curvas de depreciación por marca, el costo de los
kilómetros, las diferencias regionales y el hallazgo del "catálogo" en motos.

**Este cuaderno no implementa nada.** Toda la lógica vive en `autovalor.analysis` y aquí
solo se llama y se mira, que es la regla del proyecto: un número que aparezca en el
cuaderno tiene que poder reproducirse con `make results` sin abrir Jupyter.

Los resultados publicados están en `docs/results/`. Este cuaderno los recalcula sobre el
lago local, así que si el lago creció desde la última exportación, los números van a
diferir — y eso es justamente para lo que sirve volver a correrlo.

In [ ]:
import pandas as pd

from autovalor.analysis import catalogue, depreciation, mileage, regional, segments
from autovalor.models.dataset import load_gold, load_split
from autovalor.models.train import train_model

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

cars = load_gold("car")
bikes = load_gold("motorcycle")
len(cars), len(bikes)

## 1. Depreciación por marca

`annual_depreciation_pct` es el número de producto: incluye los kilómetros que trae un
año típico. `age_only_pct` es el mismo efecto con el kilometraje congelado. La diferencia
entre los dos es lo que cuesta usar el vehículo, separado de lo que cuesta que pase el
tiempo.

In [ ]:
car_curves = depreciation.depreciation_curves(cars, vehicle_type="car")
car_curves[
    ["brand", "listings", "annual_depreciation_pct", "ci_low_pct", "ci_high_pct", "age_only_pct", "half_life_years"]
]

In [ ]:
bike_curves = depreciation.depreciation_curves(bikes, vehicle_type="motorcycle")
bike_curves[["brand", "listings", "annual_depreciation_pct", "ci_low_pct", "ci_high_pct", "significant"]]

In [ ]:
# Cuánto vale un vehículo a cada edad, como porcentaje de uno nuevo.
depreciation.depreciation_at_ages(car_curves).pivot(
    index="brand", columns="age_years", values="retained_value_pct"
)

## 2. El costo de 10.000 km

El efecto se evalúa en la mediana de cada segmento y la fila reporta esas medianas: la
cifra en pesos solo es cierta cerca de ellas. Los segmentos se cortan por la **clase de
precio del vehículo** (la mediana de su marca y modelo), no por el precio del anuncio —
cortar por el precio pedido sería condicionar sobre la variable dependiente.

In [ ]:
pd.concat(
    [
        mileage.mileage_effects(cars, vehicle_type="car"),
        mileage.mileage_effects(bikes, vehicle_type="motorcycle"),
    ],
    ignore_index=True,
)[["vehicle_type", "segment", "n", "median_price_cop", "median_mileage_km", "cop_per_step", "significant"]]

## 3. Diferencias regionales

Cada efecto es contra el departamento de referencia (el de más anuncios), para un
vehículo comparable: mismo modelo, misma edad, mismo kilometraje. Los departamentos con
menos de 30 anuncios quedan agrupados y no se reportan, igual que el bucket de
departamento desconocido: siguen en la regresión como control, pero no son un lugar.

In [ ]:
pd.concat(
    [
        regional.regional_effects(cars, vehicle_type="car"),
        regional.regional_effects(bikes, vehicle_type="motorcycle"),
    ],
    ignore_index=True,
)[["vehicle_type", "department", "listings", "pct_change", "pct_ci_low", "pct_ci_high", "significant"]]

## 4. El hallazgo del catálogo, contrastado

SHAP dice que el modelo de motos se apoya en identidad (marca, cilindrada, modelo) y casi
no en estado (edad, kilometraje). Esa lectura tiene dos explicaciones posibles y SHAP sola
no las separa: que el estado de verdad importe poco, o que marca y modelo ya lo hayan
absorbido. El hedónico pregunta lo mismo por otra vía, con la edad puesta explícitamente.

El ajuste de abajo usa `--trials 0` para que el cuaderno corra en minutos; las cifras
publicadas salen del ajuste afinado de `make results`.

In [ ]:
importances = {}
datasets = {}
results = {}
for vehicle_type in ("car", "motorcycle"):
    dataset = load_split(vehicle_type)
    result = train_model(dataset, "lightgbm", n_trials=0, intervals=False, explain=True)
    datasets[vehicle_type] = dataset
    results[vehicle_type] = result
    importances[vehicle_type] = result.importance

catalogue.catalogue_contrast(importances, {"car": cars, "motorcycle": bikes})

## 5. Dónde se equivoca el modelo

Un MAPE por vertical es un promedio sobre un mercado que no es uniforme. Esto lo abre por
marca, por segmento de precio y por departamento, siempre sobre el holdout.

In [ ]:
errors = pd.concat(
    [
        segments.segment_errors(results[vehicle_type], datasets[vehicle_type])
        for vehicle_type in ("car", "motorcycle")
    ],
    ignore_index=True,
)
segments.worst_segments(errors, k=5)

---

Para regenerar `docs/results/` y `docs/figures/` con las cifras afinadas:

```powershell
.\make.ps1 results
```